In [ ]:
!pip install -q sentence-transformers

# **Phase 1: Data Cleaning & Integration**

In this phase, we construct a unified and clean dataset by integrating multiple heterogeneous data sources: MovieLens (user interaction data), TMDB (movie metadata), and user-generated tags.

**Objective**

The goal of this phase is to transform raw, disjoint datasets into a single structured dataset where each movie is represented with:

A unique identifier (movieId)

1. Semantic information (plot overview)

2. Categorical metadata (genres, keywords)

3. Popularity and rating statistics

4. User-generated tags





This unified representation serves as the foundation for all downstream tasks such as embedding generation, clustering, and similarity search.

**Dataset Integration**

The MovieLens and TMDB datasets do not share a common identifier. To resolve this, we use the links.csv file as a mapping bridge:



*movieId (MovieLens) → tmdbId → TMDB metadata*



We perform the following steps:



*   Remove entries with missing tmdbId
*   Convert tmdbId to integer for consistency
*   Rename TMDB's id column to tmdbId

Merge:
*   MovieLens movies with links (on movieId)
*   Result with TMDB data (on tmdbId)


This results in a dataset where each movie contains both interaction identifiers and rich metadata.

**Data Cleaning**

After merging, the dataset contains redundant and noisy columns due to overlapping schemas. We perform:

*Column renaming*:

title_x → title

genres_x → genres

Removal of duplicate/unnecessary columns:

title_y, genres_y, homepage, production_companies, etc.

Selection of relevant features:
movieId, title, overview, genres, keywords, popularity, vote_average, vote_count

**Text Preprocessing**

To prepare data for NLP-based embeddings:

Remove rows with missing overview

Normalize text by converting to lowercase

data = data.dropna(subset=["overview"])

data["overview"] = data["overview"].str.lower()


**Tag Aggregation**

The tags.csv dataset contains multiple user-generated tags per movie. These are aggregated into a single textual representation per movie:

tags["tag"] = tags["tag"].fillna("").astype(str)

tag_text = tags.groupby("movieId")["tag"].apply(
    lambda x: " ".join(x)
).reset_index()

This transforms:

["funny", "pixar", "kids"] → "funny pixar kids"

The aggregated tags are then merged with the main dataset.


**Feature Construction**

We construct a final textual feature:

data["combined_text"] = data["overview"] + " " + data["tag"]

This combines:

Plot summaries (semantic meaning)
User-generated tags (behavioral semantics)

This will later be used for transformer-based embeddings (BERT).


In [ ]:
import pandas as pd

movies = pd.read_csv("/content/drive/MyDrive/CINEFUSION/ml-25m/movies.csv")
ratings = pd.read_csv("/content/drive/MyDrive/CINEFUSION/ml-25m/ratings.csv")
links = pd.read_csv("/content/drive/MyDrive/CINEFUSION/ml-25m/links.csv")
tmdb = pd.read_csv("/content/drive/MyDrive/CINEFUSION/tmdb_5000_movies.csv")
tags = pd.read_csv("/content/drive/MyDrive/CINEFUSION/ml-25m/tags.csv")
genome_scores = pd.read_csv("/content/drive/MyDrive/CINEFUSION/ml-25m/genome-scores.csv")
genome_tags = pd.read_csv("/content/drive/MyDrive/CINEFUSION/ml-25m/genome-tags.csv")

In [ ]:
links = links.dropna(subset=["tmdbId"])
links["tmdbId"] = links["tmdbId"].astype(int)

tmdb = tmdb.rename(columns={"id": "tmdbId"})


In [ ]:
data = pd.merge(movies, links, on="movieId")
data = pd.merge(data, tmdb, on="tmdbId")

In [ ]:
data

,movieId,title_x,genres_x,imdbId,tmdbId,budget,genres_y,homepage,keywords,original_language,...,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title_y,vote_average,vote_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862,30000000,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 35, ""...",http://toystory.disney.com/toy-story,"[{""id"": 931, ""name"": ""jealousy""}, {""id"": 4290,...",en,...,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1995-10-30,373554033,81.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,Toy Story,7.7,5269
1,10,GoldenEye (1995),Action|Adventure|Thriller,113189,710,58000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 28, ""...",http://www.mgm.com/view/movie/757/Goldeneye/,"[{""id"": 701, ""name"": ""cuba""}, {""id"": 769, ""nam...",en,...,"[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",1995-11-16,352194034,130.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,No limits. No fears. No substitutes.,GoldenEye,6.6,1174
2,11,"American President, The (1995)",Comedy|Drama|Romance,112346,9087,62000000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",NaN,"[{""id"": 833, ""name"": ""white house""}, {""id"": 84...",en,...,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1995-11-17,107879496,106.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Why can't the most powerful man in the world h...,The American President,6.5,195
3,14,Nixon (1995),Drama,113987,10858,44000000,"[{""id"": 36, ""name"": ""History""}, {""id"": 18, ""na...",NaN,"[{""id"": 840, ""name"": ""usa president""}, {""id"": ...",en,...,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1995-12-22,13681765,192.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Triumphant in Victory, Bitter in Defeat. He Ch...",Nixon,7.1,71
4,15,Cutthroat Island (1995),Action|Adventure|Romance,112760,1408,98000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",NaN,"[{""id"": 911, ""name"": ""exotic island""}, {""id"": ...",en,...,"[{""iso_3166_1"": ""FR"", ""name"": ""France""}, {""iso...",1995-12-22,10017322,119.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,The Course Has Been Set. There Is No Turning B...,Cutthroat Island,5.7,136
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4597,200562,A Fine Step (2014),Drama,1604100,248402,0,"[{""id"": 18, ""name"": ""Drama""}]",NaN,[],en,...,[],2014-04-16,0,90.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Together they learnt to dream again,A Fine Step,4.1,7
4598,201050,Zombie Hunter (2013),Action|Comedy|Sci-Fi|Thriller,2446502,206213,0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 28, ""nam...",https://www.facebook.com/zombiehuntermovie,"[{""id"": 1852, ""name"": ""mutant""}, {""id"": 4458, ...",en,...,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2013-07-25,0,93.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,Zombie Hunter,3.5,34
4599,203797,Excessive Force (1993),Action,104215,24227,3000000,"[{""id"": 28, ""name"": ""Action""}]",NaN,"[{""id"": 6149, ""name"": ""police""}, {""id"": 181508...",en,...,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1993-05-14,1200000,87.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,You have the right to remain silent... permane...,Excessive Force,4.5,10
4600,204288,Open Secret (1948),Crime|Mystery|Thriller,40671,51130,0,"[{""id"": 80, ""name"": ""Crime""}, {""id"": 9648, ""na...",NaN,"[{""id"": 9937, ""name"": ""suspense""}, {""id"": 1954...",en,...,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1948-02-14,0,68.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Pull-No-Punch drama of men chained togethe...,Open Secret,7.0,2


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4602 entries, 0 to 4601
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movieId               4602 non-null   int64  
 1   title_x               4602 non-null   object 
 2   genres_x              4602 non-null   object 
 3   imdbId                4602 non-null   int64  
 4   tmdbId                4602 non-null   int64  
 5   budget                4602 non-null   int64  
 6   genres_y              4602 non-null   object 
 7   homepage              1658 non-null   object 
 8   keywords              4602 non-null   object 
 9   original_language     4602 non-null   object 
 10  original_title        4602 non-null   object 
 11  overview              4601 non-null   object 
 12  popularity            4602 non-null   float64
 13  production_companies  4602 non-null   object 
 14  production_countries  4602 non-null   object 
 15  release_date         

In [ ]:
data = data.rename(columns={
    "title_x": "title",
    "genres_x": "genres"
})


In [ ]:

drop_cols = [
    "title_y",
    "genres_y",
    "homepage",
    "production_companies",
    "production_countries",
    "spoken_languages",
    "status",
    "tagline",
    "original_title"
]

data = data.drop(columns=[col for col in drop_cols if col in data.columns])

In [ ]:
data = data[[
    "movieId",
    "title",
    "overview",
    "genres",
    "keywords",
    "popularity",
    "vote_average",
    "vote_count"
]]

In [ ]:
data

,movieId,title,overview,genres,keywords,popularity,vote_average,vote_count
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Adventure|Animation|Children|Comedy|Fantasy,"[{""id"": 931, ""name"": ""jealousy""}, {""id"": 4290,...",73.640445,7.7,5269
1,10,GoldenEye (1995),James Bond must unmask the mysterious head of ...,Action|Adventure|Thriller,"[{""id"": 701, ""name"": ""cuba""}, {""id"": 769, ""nam...",59.824565,6.6,1174
2,11,"American President, The (1995)","Widowed U.S. president Andrew Shepherd, one of...",Comedy|Drama|Romance,"[{""id"": 833, ""name"": ""white house""}, {""id"": 84...",11.056763,6.5,195
3,14,Nixon (1995),An all-star cast powers this epic look at Amer...,Drama,"[{""id"": 840, ""name"": ""usa president""}, {""id"": ...",3.770161,7.1,71
4,15,Cutthroat Island (1995),"Morgan Adams and her slave, William Shaw, are ...",Action|Adventure|Romance,"[{""id"": 911, ""name"": ""exotic island""}, {""id"": ...",7.029308,5.7,136
...,...,...,...,...,...,...,...,...
4597,200562,A Fine Step (2014),A Fine Step is an uplifting family drama cente...,Drama,[],0.654340,4.1,7
4598,201050,Zombie Hunter (2013),Zombie Hunter is set in a post-apocalyptic Zom...,Action|Comedy|Sci-Fi|Thriller,"[{""id"": 1852, ""name"": ""mutant""}, {""id"": 4458, ...",3.418372,3.5,34
4599,203797,Excessive Force (1993),Chicago policeman Terry McCain is determined t...,Action,"[{""id"": 6149, ""name"": ""police""}, {""id"": 181508...",1.279106,4.5,10
4600,204288,Open Secret (1948),A couple discovers that their friend has gone ...,Crime|Mystery|Thriller,"[{""id"": 9937, ""name"": ""suspense""}, {""id"": 1954...",0.186401,7.0,2


In [ ]:
# Remove null values in  overview
data = data.dropna(subset=["overview"])

# Normalized the text
data["overview"] = data["overview"].str.lower()

/tmp/ipykernel_1683/3373274305.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["overview"] = data["overview"].str.lower()


In [ ]:
data

,movieId,title,overview,genres,keywords,popularity,vote_average,vote_count
0,1,Toy Story (1995),"led by woody, andy's toys live happily in his ...",Adventure|Animation|Children|Comedy|Fantasy,"[{""id"": 931, ""name"": ""jealousy""}, {""id"": 4290,...",73.640445,7.7,5269
1,10,GoldenEye (1995),james bond must unmask the mysterious head of ...,Action|Adventure|Thriller,"[{""id"": 701, ""name"": ""cuba""}, {""id"": 769, ""nam...",59.824565,6.6,1174
2,11,"American President, The (1995)","widowed u.s. president andrew shepherd, one of...",Comedy|Drama|Romance,"[{""id"": 833, ""name"": ""white house""}, {""id"": 84...",11.056763,6.5,195
3,14,Nixon (1995),an all-star cast powers this epic look at amer...,Drama,"[{""id"": 840, ""name"": ""usa president""}, {""id"": ...",3.770161,7.1,71
4,15,Cutthroat Island (1995),"morgan adams and her slave, william shaw, are ...",Action|Adventure|Romance,"[{""id"": 911, ""name"": ""exotic island""}, {""id"": ...",7.029308,5.7,136
...,...,...,...,...,...,...,...,...
4597,200562,A Fine Step (2014),a fine step is an uplifting family drama cente...,Drama,[],0.654340,4.1,7
4598,201050,Zombie Hunter (2013),zombie hunter is set in a post-apocalyptic zom...,Action|Comedy|Sci-Fi|Thriller,"[{""id"": 1852, ""name"": ""mutant""}, {""id"": 4458, ...",3.418372,3.5,34
4599,203797,Excessive Force (1993),chicago policeman terry mccain is determined t...,Action,"[{""id"": 6149, ""name"": ""police""}, {""id"": 181508...",1.279106,4.5,10
4600,204288,Open Secret (1948),a couple discovers that their friend has gone ...,Crime|Mystery|Thriller,"[{""id"": 9937, ""name"": ""suspense""}, {""id"": 1954...",0.186401,7.0,2


In [ ]:
# Cleaned the tag column
tags["tag"] = tags["tag"].fillna("").astype(str)
tag_text = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x)).reset_index()
print("Tags:",tags)
print("Tag Text:",tag_text)

Tags:          userId  movieId                  tag   timestamp
0             3      260              classic  1439472355
1             3      260               sci-fi  1439472256
2             4     1732          dark comedy  1573943598
3             4     1732       great dialogue  1573943604
4             4     7569     so bad it's good  1573943455
...         ...      ...                  ...         ...
1093355  162521    66934  Neil Patrick Harris  1427311611
1093356  162521   103341     cornetto trilogy  1427311259
1093357  162534   189169               comedy  1527518175
1093358  162534   189169             disabled  1527518181
1093359  162534   189169              robbery  1527518193

[1093360 rows x 4 columns]
Tag Text:        movieId                                                tag
0            1  Owned imdb top 250 Pixar Pixar time travel chi...
1            2  Robin Williams time travel fantasy based on ch...
2            3  funny best friend duringcreditsstinger fishing

In [ ]:
data = pd.merge(data, tag_text, on="movieId", how="left")
data["tag"] = data["tag"].fillna("")

In [ ]:
data

,movieId,title,overview,genres,keywords,popularity,vote_average,vote_count,tag
0,1,Toy Story (1995),"led by woody, andy's toys live happily in his ...",Adventure|Animation|Children|Comedy|Fantasy,"[{""id"": 931, ""name"": ""jealousy""}, {""id"": 4290,...",73.640445,7.7,5269,Owned imdb top 250 Pixar Pixar time travel chi...
1,10,GoldenEye (1995),james bond must unmask the mysterious head of ...,Action|Adventure|Thriller,"[{""id"": 701, ""name"": ""cuba""}, {""id"": 769, ""nam...",59.824565,6.6,1174,007 Bond boys with toys gadgets secret service...
2,11,"American President, The (1995)","widowed u.s. president andrew shepherd, one of...",Comedy|Drama|Romance,"[{""id"": 833, ""name"": ""white house""}, {""id"": 84...",11.056763,6.5,195,Romance white house new love usa president whi...
3,14,Nixon (1995),an all-star cast powers this epic look at amer...,Drama,"[{""id"": 840, ""name"": ""usa president""}, {""id"": ...",3.770161,7.1,71,biography government historical figure preside...
4,15,Cutthroat Island (1995),"morgan adams and her slave, william shaw, are ...",Action|Adventure|Romance,"[{""id"": 911, ""name"": ""exotic island""}, {""id"": ...",7.029308,5.7,136,exotic island map pirate scalp ship treasure b...
...,...,...,...,...,...,...,...,...,...
4596,200562,A Fine Step (2014),a fine step is an uplifting family drama cente...,Drama,[],0.654340,4.1,7,
4597,201050,Zombie Hunter (2013),zombie hunter is set in a post-apocalyptic zom...,Action|Comedy|Sci-Fi|Thriller,"[{""id"": 1852, ""name"": ""mutant""}, {""id"": 4458, ...",3.418372,3.5,34,
4598,203797,Excessive Force (1993),chicago policeman terry mccain is determined t...,Action,"[{""id"": 6149, ""name"": ""police""}, {""id"": 181508...",1.279106,4.5,10,
4599,204288,Open Secret (1948),a couple discovers that their friend has gone ...,Crime|Mystery|Thriller,"[{""id"": 9937, ""name"": ""suspense""}, {""id"": 1954...",0.186401,7.0,2,


In [ ]:
data["combined_text"] = (
    data["overview"] + " " + data["tag"]
)

In [ ]:
data.head()

,movieId,title,overview,genres,keywords,popularity,vote_average,vote_count,tag,combined_text
0,1,Toy Story (1995),"led by woody, andy's toys live happily in his ...",Adventure|Animation|Children|Comedy|Fantasy,"[{""id"": 931, ""name"": ""jealousy""}, {""id"": 4290,...",73.640445,7.7,5269,Owned imdb top 250 Pixar Pixar time travel chi...,"led by woody, andy's toys live happily in his ..."
1,10,GoldenEye (1995),james bond must unmask the mysterious head of ...,Action|Adventure|Thriller,"[{""id"": 701, ""name"": ""cuba""}, {""id"": 769, ""nam...",59.824565,6.6,1174,007 Bond boys with toys gadgets secret service...,james bond must unmask the mysterious head of ...
2,11,"American President, The (1995)","widowed u.s. president andrew shepherd, one of...",Comedy|Drama|Romance,"[{""id"": 833, ""name"": ""white house""}, {""id"": 84...",11.056763,6.5,195,Romance white house new love usa president whi...,"widowed u.s. president andrew shepherd, one of..."
3,14,Nixon (1995),an all-star cast powers this epic look at amer...,Drama,"[{""id"": 840, ""name"": ""usa president""}, {""id"": ...",3.770161,7.1,71,biography government historical figure preside...,an all-star cast powers this epic look at amer...
4,15,Cutthroat Island (1995),"morgan adams and her slave, william shaw, are ...",Action|Adventure|Romance,"[{""id"": 911, ""name"": ""exotic island""}, {""id"": ...",7.029308,5.7,136,exotic island map pirate scalp ship treasure b...,"morgan adams and her slave, william shaw, are ..."


In [ ]:
data["genres"]

,genres
0,Adventure|Animation|Children|Comedy|Fantasy
1,Action|Adventure|Thriller
2,Comedy|Drama|Romance
3,Drama
4,Action|Adventure|Romance
...,...
4596,Drama
4597,Action|Comedy|Sci-Fi|Thriller
4598,Action
4599,Crime|Mystery|Thriller


# **Phase 2: ALS**

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CineFusion").getOrCreate()

In [ ]:
ratings_spark = spark.read.csv(
    "/content/drive/MyDrive/CINEFUSION/ml-25m/ratings.csv",
    header=True,
    inferSchema=True
)

In [ ]:

movie_ids = spark.createDataFrame(data[["movieId"]])

ratings_spark = ratings_spark.join(movie_ids, on="movieId")

print("Filtered Spark ratings:", ratings_spark.count())

Filtered Spark ratings: 17258133


In [ ]:
ratings_spark.select("movieId").distinct().count()

4594

In [ ]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=50,
    maxIter=5,
    regParam=0.01,
    coldStartStrategy="drop"
)

model = als.fit(ratings_spark)

In [ ]:
item_factors = model.itemFactors.toPandas()

print(item_factors.head())

   id                                           features
0  10  [0.09051796793937683, -0.166781485080719, 0.01...
1  20  [0.14035654067993164, 0.26848363876342773, 0.2...
2  50  [0.12229345738887787, 0.15548191964626312, 0.2...
3  60  [-0.1459018588066101, 0.4940400719642639, 0.05...
4  70  [-0.02218530885875225, 0.31569910049438477, 0....


In [ ]:
# Renamed the columns
item_factors = item_factors.rename(columns={"id": "movieId"})

# Converted the embeddings
cf_embeddings = pd.DataFrame(
    item_factors["features"].to_list(),
    columns=[f"cf_{i}" for i in range(50)]
)

cf_embeddings["movieId"] = item_factors["movieId"]

data = pd.merge(data, cf_embeddings, on="movieId", how="inner")

print("Final data shape after ALS:", data.shape)

Final data shape after ALS: (4594, 110)


In [ ]:
data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4594 entries, 0 to 4593
Columns: 110 entries, movieId to cf_49
dtypes: float64(102), int64(2), object(6)
memory usage: 3.9+ MB


In [ ]:
data

,movieId,title,overview,genres,keywords,popularity,vote_average,vote_count,tag,combined_text,...,cf_40,cf_41,cf_42,cf_43,cf_44,cf_45,cf_46,cf_47,cf_48,cf_49
0,1,Toy Story (1995),"led by woody, andy's toys live happily in his ...",Adventure|Animation|Children|Comedy|Fantasy,"[{""id"": 931, ""name"": ""jealousy""}, {""id"": 4290,...",73.640445,7.7,5269,Owned imdb top 250 Pixar Pixar time travel chi...,"led by woody, andy's toys live happily in his ...",...,-0.182601,-0.135772,0.187250,0.017958,-0.114649,-0.076015,0.335595,0.111449,0.079439,0.152711
1,10,GoldenEye (1995),james bond must unmask the mysterious head of ...,Action|Adventure|Thriller,"[{""id"": 701, ""name"": ""cuba""}, {""id"": 769, ""nam...",59.824565,6.6,1174,007 Bond boys with toys gadgets secret service...,james bond must unmask the mysterious head of ...,...,-0.308101,0.187883,-0.320966,0.256050,-0.175228,-0.200106,0.401932,0.257480,-0.455210,0.190841
2,11,"American President, The (1995)","widowed u.s. president andrew shepherd, one of...",Comedy|Drama|Romance,"[{""id"": 833, ""name"": ""white house""}, {""id"": 84...",11.056763,6.5,195,Romance white house new love usa president whi...,"widowed u.s. president andrew shepherd, one of...",...,-0.374762,0.415817,0.228878,0.167397,-0.099243,0.299265,0.035022,0.017630,-0.374594,0.210945
3,14,Nixon (1995),an all-star cast powers this epic look at amer...,Drama,"[{""id"": 840, ""name"": ""usa president""}, {""id"": ...",3.770161,7.1,71,biography government historical figure preside...,an all-star cast powers this epic look at amer...,...,0.244392,0.703777,0.176137,-0.036567,0.135155,-0.117663,-0.068716,-0.166492,0.080811,0.285713
4,15,Cutthroat Island (1995),"morgan adams and her slave, william shaw, are ...",Action|Adventure|Romance,"[{""id"": 911, ""name"": ""exotic island""}, {""id"": ...",7.029308,5.7,136,exotic island map pirate scalp ship treasure b...,"morgan adams and her slave, william shaw, are ...",...,-0.369031,0.117033,-0.165545,0.133833,-0.328396,-0.337225,0.122736,0.353513,-0.396174,-0.138309
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4589,200562,A Fine Step (2014),a fine step is an uplifting family drama cente...,Drama,[],0.654340,4.1,7,,a fine step is an uplifting family drama cente...,...,-0.004142,0.140156,-0.046597,0.150033,-0.149580,-0.107173,0.119188,-0.001613,0.030402,0.032640
4590,201050,Zombie Hunter (2013),zombie hunter is set in a post-apocalyptic zom...,Action|Comedy|Sci-Fi|Thriller,"[{""id"": 1852, ""name"": ""mutant""}, {""id"": 4458, ...",3.418372,3.5,34,,zombie hunter is set in a post-apocalyptic zom...,...,0.040598,-0.033094,-0.167046,0.195553,-0.230679,-0.199945,0.135792,-0.001294,0.125338,-0.068301
4591,203797,Excessive Force (1993),chicago policeman terry mccain is determined t...,Action,"[{""id"": 6149, ""name"": ""police""}, {""id"": 181508...",1.279106,4.5,10,,chicago policeman terry mccain is determined t...,...,-0.070425,0.326565,0.049383,0.054023,-0.065444,0.200141,0.047400,0.138298,-0.088709,0.164136
4592,204288,Open Secret (1948),a couple discovers that their friend has gone ...,Crime|Mystery|Thriller,"[{""id"": 9937, ""name"": ""suspense""}, {""id"": 1954...",0.186401,7.0,2,,a couple discovers that their friend has gone ...,...,0.102833,0.201637,-0.024443,0.203899,-0.228759,-0.174072,0.156686,0.071950,-0.186496,0.214951


## **BUILDING USER PROFILE BASED ON WATCH HISTORY**

In [ ]:
model.save("/content/als_model")

In [ ]:
model.itemFactors.write.mode("overwrite").parquet("/content/movie_embeddings")

In [ ]:
from pyspark.ml.recommendation import ALSModel

model = ALSModel.load("/content/als_model")

In [ ]:
item_factors = spark.read.parquet("/content/movie_embeddings").toPandas()
item_factors = item_factors.rename(columns={"id": "movieId"})

In [ ]:
import numpy as np
from sklearn.preprocessing import normalize

cf_cols = [f"cf_{i}" for i in range(50)]

def recommend_movies(user_id, ratings_df, data_df, top_k=10):
    """
    Production-ready recommender
    """

    # 1. Get user history
    user_history = ratings_df[ratings_df["userId"] == user_id]

    if user_history.shape[0] == 0:
        print("New user: no history found")
        return None

    # 2. Get user movies
    user_movies = data_df[data_df["movieId"].isin(user_history["movieId"])]

    # 3. Build user vector (weighted)
    movie_vectors = user_movies[cf_cols].values

    ratings = user_history.set_index("movieId").loc[user_movies["movieId"]]["rating"].values
    user_vector = (movie_vectors * ratings[:, np.newaxis]).mean(axis=0)

    # Normalize
    user_vector = normalize(user_vector.reshape(1, -1))[0]

    # 4. Compute scores (vectorized FAST)
    movie_matrix = data_df[cf_cols].values
    scores = movie_matrix @ user_vector

    data_df["score"] = scores

    # 5. Remove seen movies
    seen = set(user_history["movieId"])
    recommendations = data_df[~data_df["movieId"].isin(seen)]

    # 6. Top-K
    return recommendations.sort_values("score", ascending=False)[
        ["movieId", "title", "score"]
    ].head(top_k)

In [ ]:
recommend_movies(user_id=1, ratings_df=ratings, data_df=data, top_k=10)

,movieId,title,score
1820,6981,"Ordet (Word, The) (1955)",1.237959
2075,27664,"Brown Bunny, The (2003)",1.205765
2787,59684,Lake of Fire (2006),1.186478
2137,31900,Travellers and Magicians (2003),1.165756
1756,6682,Earth (1998),1.153052
2010,8938,Tarnation (2003),1.152302
516,1916,Buffalo '66 (a.k.a. Buffalo 66) (1998),1.141768
1635,6053,Krush Groove (1985),1.138515
1920,7814,Waterloo (1970),1.136368
1589,5817,Ararat (2002),1.124810
